# SchoolPay — Enhanced Entity-Relationship (EER) Diagram

**Version:** 1.0  
**Date:** June 2026  
**Scope:** Ugandan secondary school (S1–S4) management system  
**Author:** Opiyo Oscar  
**Reg No:** 23/U/1330  
**Student No:** 2300701330  
**Track:** Web Development  
**Canonical doc:** [eerd-diagram.ipynb](./eerd-diagram.ipynb) (this notebook)  
**Related:** [schema.sql](../schema.sql), [relationship.dbml](../docx/relationship.dbml)

---
## 1. Entity catalog

The system needs to remember **29 entities** — 18 currently implemented in SQL, 11 planned.

### Reference / Lookup entities (rarely change)

| # | Entity | Description | Status |
|---|--------|-------------|--------|
| 1 | `academic_years` | School calendar years (e.g. 2025, 2026) | 📋 Planned |
| 2 | `terms` | Term divisions within an academic year (Term 1–3) | ✅ Existing |
| 3 | `classes` | Class levels with stream (e.g. S1 East, S2 West) | ✅ Existing |
| 4 | `subjects` | Academic subjects (Math, English, Biology, ...) | ✅ Existing |
| 5 | `fee_types` | Fee categories (Tuition, Development, PTA, ...) | ✅ Existing |
| 6 | `books` | Library book inventory | ✅ Existing |
| 7 | `staff_roles` | Role definitions (Teacher, Bursar, Admin, Head Teacher) | 📋 Planned |
| 8 | `generic_skills` | Soft/skill categories for termly assessment | 📋 Planned |

### Transactional entities (constantly change)

| # | Entity | Description | Status |
|---|--------|-------------|--------|
| 9 | `students` | Learner enrollment records | ✅ Existing |
| 10 | `student_contacts` | Guardian/parent/emergency contacts per student | ✅ Existing |
| 11 | `guardians` | Independent guardian entity (shared across siblings) | 📋 Planned |
| 12 | `staff` | Teaching and non-teaching staff | ✅ Existing |
| 13 | `class_teacher` | Assigns a teacher as class teacher for a term | ✅ Existing |
| 14 | `subject_teacher` | Assigns a teacher to teach a subject in a class for a term | ✅ Existing |
| 15 | `fee_structure` | Fee amounts per class × fee_type × term | ✅ Existing |
| 16 | `payments` | Payment transactions made by students | ✅ Existing |
| 17 | `payment_receipts` | Receipts generated for payments | ✅ Existing |
| 18 | `attendance` | Daily attendance records per student | ✅ Existing |
| 19 | `exams` | Exam event definitions per term | ✅ Existing |
| 20 | `exam_results` | Per-student, per-subject marks and grades | ✅ Existing |
| 21 | `book_loans` | Book borrowing records | ✅ Existing |
| 22 | `notices` | System-wide announcements | ✅ Existing |
| 23 | `enrollments` | Student enrollment in an academic year | 📋 Planned |
| 24 | `discipline_records` | Student discipline incidents | 📋 Planned |
| 25 | `skill_assessments` | Per-student, per-term generic skill ratings | 📋 Planned |
| 26 | `promotion_records` | Student promotion to next class | 📋 Planned |
| 27 | `repetition_records` | Student repeating a class | 📋 Planned |
| 28 | `transfer_records` | Student transfers in/out | 📋 Planned |
| 29 | `timetable_slots` | Class-teacher-subject-period schedule | 📋 Planned |

---
## 2. EER specializations & generalizations

### 2.1 Specialization: Staff (by role)

```mermaid
graph TD
    Staff[Staff] -->|role = Teacher| Teacher
    Staff -->|role = Bursar| Bursar
    Staff -->|role = Admin| Admin
    Staff -->|role = Head Teacher| HeadTeacher[Head Teacher]
```

**Disjoint, attribute-defined** — single `role` column distinguishes all subtypes.

### 2.2 Specialization: TeachingStaff vs NonTeachingStaff

```mermaid
graph TD
    Staff[Staff] -->|teaches?| Teaching[Teaching Staff]
    Staff -->|teaches?| NonTeaching[Non-Teaching Staff]
    NonTeaching --> Bursar
    NonTeaching --> Admin
    NonTeaching --> HeadTeacher[Head Teacher]
```

### 2.3 Specialization: Exam (by exam_type)

```mermaid
graph TD
    Exam[Exam] -->|type = End of Term| EndOfTerm
    Exam -->|type = Continuous Assessment| CA[Continuous Assessment]
    Exam -->|type = Mock| Mock
    Exam -->|type = UNEB| UNEB
```

### 2.4 Category (Union): BookLoan Borrower

```mermaid
graph LR
    Student -->|borrows| BookLoan[BookLoan]
    Staff -->|borrows| BookLoan
```

A BookLoan may be borrowed by **either** a Student **or** a Staff member (union type). Current schema only supports student borrowers; staff borrower support via `staff_borrower_id` is planned.

---
## 3. Entity-Relationship diagram

```mermaid
erDiagram
    academic_years ||--o{ terms : contains
    terms ||--o{ class_teacher : defines
    terms ||--o{ subject_teacher : defines
    terms ||--o{ fee_structure : applies
    terms ||--o{ exams : schedules
    terms ||--o{ attendance : tracks
    terms ||--o{ skill_assessments : evaluates
    terms ||--o{ enrollments : belongs_to

    classes ||--o{ students : enrolls
    classes ||--o{ class_teacher : assigned
    classes ||--o{ subject_teacher : assigned
    classes ||--o{ fee_structure : has_fees
    classes ||--o{ timetable_slots : schedules

    students ||--o{ student_contacts : has
    students ||--o{ enrollments : enrolls
    students ||--o{ payments : makes
    students ||--o{ attendance : records
    students ||--o{ exam_results : achieves
    students ||--o{ book_loans : borrows
    students ||--o{ discipline_records : involves
    students ||--o{ skill_assessments : assesses
    students ||--o{ promotion_records : promotes
    students ||--o{ repetition_records : repeats
    students ||--o{ transfer_records : transfers

    students }o--|| guardians : "student_guardian (M:N)"
    guardians ||--o{ student_contacts : listed_as

    staff ||--o{ class_teacher : assigned_as
    staff ||--o{ subject_teacher : teaches
    staff ||--o{ payment_receipts : issues
    staff ||--o{ attendance : recorded_by
    staff ||--o{ notices : posts
    staff ||--o{ timetable_slots : assigned

    staff }o--|| staff_roles : has_role

    subjects ||--o{ subject_teacher : assigned
    subjects ||--o{ exam_results : assessed_in
    subjects ||--o{ books : categorized

    fee_types ||--o{ fee_structure : categorized_as
    fee_structure ||--o{ payments : billed_against
    payments ||--|| payment_receipts : generates

    exams ||--o{ exam_results : produces
    books ||--o{ book_loans : loaned
    generic_skills ||--o{ skill_assessments : rated_in
```

**Rendered image:** [erd-draft.png](./erd-draft.png)

---
## 4. Relationship matrix

| Entity A | Relationship | Entity B | Cardinality | Concrete example |
|----------|-------------|----------|-------------|------------------|
| academic_year | contains | term | 1:N | Year 2025 → Term 1, Term 2, Term 3 |
| term | defines | class_teacher | 1:N | Term 1 assigns class teachers for that term |
| term | schedules | exam | 1:N | Term 1 → Mid-Term, End-Term exams |
| class | enrolls | student | 1:N | S1 East → 45 students |
| class | has | fee_structure | 1:N | S1 East → Tuition=300k, Dev=50k |
| student | has | student_contact | 1:N | John → Mother (emergency), Father |
| student | makes | payment | 1:N | John → 3 payments in Term 1 |
| student | achieves | exam_result | 1:N | John → 8 exam results (1 per subject) |
| student | records | attendance | 1:N | John → 60 daily attendance records |
| student | borrows | book_loan | 1:N | John → 2 book loans in Term 1 |
| **student** | **<–>** | **guardian** | **M:N** | **John → Mrs. Smith → also guardian of Mary** |
| staff | assigned_as | class_teacher | 1:N | Mr. Mutebi → class teacher S1 East |
| staff | teaches | subject_teacher | 1:N | Mr. Mutebi → Math in S1 East |
| staff | has | staff_role | N:1 | 5 teachers → "Teacher" role |
| subject | assessed_in | exam_result | 1:N | Math → 45 exam results (1/student) |
| exam | produces | exam_result | 1:N | End-Term → 360 exam results |
| fee_type | categorized | fee_structure | 1:N | "Tuition" → appears in every class fee structure |
| payment | generates | payment_receipt | 1:1 | Each payment → exactly one receipt |
| book | loaned | book_loan | 1:N | "Maths for S1" → loaned 15 times |
| student | involves | discipline_record | 1:N | John → 2 discipline incidents |
| student | assesses | skill_assessment | 1:N | John → 5 skill ratings per term |
| student | promotes | promotion_record | 1:N | John → S1→S2 promotion record |
| student | repeats | repetition_record | 1:N | Mary → S1 repetition record |
| class | schedules | timetable_slot | 1:N | S1 East → 40 weekly timetable slots |

---
## 5. Reference vs Transactional entities

### Lookup / Reference (rarely changes)

| Entity | Update cadence |
|--------|---------------|
| `academic_years` | Once per year |
| `terms` | Once per term (3× per year) |
| `classes` | When new stream opens |
| `subjects` | When curriculum changes |
| `fee_types` | Created once (Tuition, PTA, etc.) |
| `books` | When new books arrive |
| `staff_roles` | Created once (Teacher, Bursar, etc.) |
| `generic_skills` | Created once (Punctuality, Teamwork, etc.) |

### Transactional (constantly changes)

| Entity | Frequency |
|--------|-----------|
| `students` | Enrolled, promoted, transferred daily |
| `payments` | Recorded daily |
| `attendance` | Recorded daily |
| `exam_results` | Per exam event (3–4× per term) |
| `book_loans` | Per borrow/return transaction |
| `discipline_records` | Per incident |
| `skill_assessments` | Once per term |
| `fee_structure` | Updated per term |
| `class_teacher` | Assigned per term |
| `subject_teacher` | Assigned per term |
| `timetable_slots` | Created per term |
| `notices` | Posted as needed |
| `promotion_records` | Year-end |
| `repetition_records` | Year-end |
| `transfer_records` | Per transfer event |

---
## 6. Entity status map

---
## 7. Entity status map

| Badge | Meaning |
|-------|---------|
| ✅ Existing | Fully implemented in current SQL schema |
| 📋 Planned | Defined in DBML but not yet implemented |

### Status per entity

| Entity | Status | Schema file(s) | Notes |
|--------|--------|----------------|-------|
| `terms` | ✅ | All schema files | Present with term_name, academic_year, dates |
| `classes` | ✅ | All schema files | With class_name, stream, level, capacity |
| `students` | ✅ | All schema files | Core entity, FK → classes |
| `student_contacts` | ✅ | All schema files | Guardian names, phone, relationship |
| `staff` | ✅ | All schema files | Single table with role ENUM |
| `class_teacher` | ✅ | All schema files | Junction: staff + class + term |
| `subjects` | ✅ | All schema files | Subject catalog with code |
| `subject_teacher` | ✅ | All schema files | Junction: staff + subject + class + term |
| `fee_types` | ✅ | All schema files | Fee category lookup |
| `fee_structure` | ✅ | All schema files | class × fee_type × term → amount |
| `payments` | ✅ | All schema files | Transactional payment records |
| `payment_receipts` | ✅ | All schema files | 1:1 with payments |
| `attendance` | ✅ | All schema files | Per-student daily records |
| `exams` | ✅ | All schema files | Exam events per term |
| `exam_results` | ✅ | All schema files | Marks per student per subject |
| `books` | ✅ | All schema files | Library inventory |
| `book_loans` | ✅ | All schema files | Borrow records (student only) |
| `notices` | ✅ | All schema files | Announcements with audience |
| `academic_years` | 📋 | `docx/relationship.dbml` | Implicit in terms; extract for clarity |
| `guardians` | 📋 | `docx/relationship.dbml` | Currently embedded in student_contacts |
| `staff_roles` | 📋 | `docx/relationship.dbml` | Currently an ENUM on staff.role |
| `generic_skills` | 📋 | `docx/relationship.dbml` | No SQL table yet |
| `skill_assessments` | 📋 | `docx/relationship.dbml` | No SQL table yet |
| `enrollments` | 📋 | `docx/relationship.dbml` | Track year-by-year enrollment |
| `discipline_records` | 📋 | `docx/relationship.dbml` | No SQL table yet |
| `promotion_records` | 📋 | `docx/relationship.dbml` | No SQL table yet |
| `repetition_records` | 📋 | `docx/relationship.dbml` | No SQL table yet |
| `transfer_records` | 📋 | `docx/relationship.dbml` | No SQL table yet |
| `timetable_slots` | 📋 | `docx/relationship.dbml` | No SQL table yet |

---
## 7. References

1. **DBML source (alternative notation):** [`docx/relationship.dbml`](../docx/relationship.dbml)
2. **Rendered ERD image:** [`docs/erd-draft.png`](./erd-draft.png)
3. **Main schema implementation:** [`schema.sql`](../schema.sql)
4. **Database setup DDL:** [`01_database_setup.sql`](../01_database_setup.sql)

---

*Notebook generated from the SchoolPay EER analysis. Update this notebook when entities are added or relationships change.*